## Gold Layer

In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

### Read Silver Table

In [0]:
df_silver = spark.read.table("fintech.silver.stock_prices")

### Update dim_stock
- For dim_stock we create stock_key as a surrogate key incremental by using row_number()

In [0]:
# =========================================================
# Get distinct symbols from Silver
# =========================================================
df_symbols = (
    df_silver
    .select("symbol")
    .distinct()
    .filter(F.col("symbol").isNotNull())
)

# =========================================================
# Check if dim_stock exists
# =========================================================
dim_stocks_exists = spark.catalog.tableExists(
    "fintech.gold.dim_stock"
)

# =========================================================
# Create dimension if it does not exist
# =========================================================
if not dim_stocks_exists:

    window = Window.orderBy("symbol")

    df_dim_stock = (
        df_symbols.withColumn(
            "stock_key", 
            F.row_number().over(window) 
        )
        .select(
            "stock_key", 
            "symbol"
        )
    )

    (
        df_dim_stock
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable("fintech.gold.dim_stock")
    )

# =========================================================
# Existing dimension
# =========================================================
else:
    df_dim_stock = (
        spark.table("fintech.gold.dim_stock")
        .select("stock_key", "symbol")
    )

    # =====================================================
    # Find new stocks
    # =====================================================
    df_new_symbols = (
        df_symbols
        .join(
            df_dim_stock.select("symbol"),
            on="symbol",
            how="left_anti"
        )
    )

    # =====================================================
    # Generate new surrogate keys
    # =====================================================
    max_key = (
        df_dim_stock
        .agg(
            F.max("stock_key").alias("max_key")
        )
        .collect()[0]["max_key"]
    )

    if max_key is None:
        max_key = 0

    window = Window.orderBy("symbol")

    df_new_symbols = (
        df_new_symbols
        .withColumn(
            "stock_key",
            F.row_number().over(window) + F.lit(max_key)
        )
        .select("stock_key", "symbol")
    )

    # =====================================================
    # Insert new stocks
    # =====================================================
    if not df_new_symbols.isEmpty():
        (
            df_new_symbols
            .write
            .format("delta")
            .mode("append")
            .saveAsTable("fintech.gold.dim_stock")
        )